# Plate 02 · Synthetic Kuwaiti Plate Generator

Real Kuwaiti plate photos are scarce and unbalanced (few taxi / commercial / night images).
This generator renders **100k+ labelled plates** in the canonical layout of `Kuwait_Plate_Layout.py`:
left strip "الكويت / KUWAIT" with the flag colours, then the **code** and the **number**, each printed
in Arabic-Indic digits (top) and English digits (bottom).

It writes two things:
1. **Cell crops** (code/number × Arabic/English) for the recogniser - always pixel-aligned with the real pipeline.
2. **Whole plates pasted on random backgrounds** with box + 4 corners for the detector (YOLO-pose labels).

Fonts (SIL OFL, from Google Fonts) into `Data/Plates/Fonts/`: an Arabic font (Noto Kufi Arabic / Noto Naskh Arabic)
and a bold sans font for English digits (e.g. Roboto / Noto Sans). Augmentations: perspective, motion blur,
night (low gamma + noise), rain streaks, glare, dirt, JPEG artefacts.

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Plate_Recognition
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import random
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from Common_Utils.Kuwait_Plate_Layout import CANONICAL_W, CANONICAL_H, CELLS, CELL_W, CELL_H, cell_box_px
from Common_Utils.Plate_Normalizer import to_arabic_digits

FONTS = DATA / "Plates" / "Fonts"
ARABIC_FONTS = sorted(FONTS.glob("*Arabic*.ttf")) or sorted(FONTS.glob("*.ttf"))
LATIN_FONTS = sorted(p for p in FONTS.glob("*.ttf") if "Arabic" not in p.name) or ARABIC_FONTS
OUT = DATA / "Plates" / "Synthetic"
(OUT / "cells").mkdir(parents=True, exist_ok=True)
(OUT / "detection" / "images").mkdir(parents=True, exist_ok=True)
(OUT / "detection" / "labels").mkdir(parents=True, exist_ok=True)
assert ARABIC_FONTS, "Put Arabic + Latin .ttf fonts in Data/Plates/Fonts/"

# plate type -> (background, text colour). CONFIRM with current MOI plate rules in Phase 0.
STYLES = {
    "private": ((248, 248, 246), (15, 15, 15)),
    "taxi": ((245, 196, 48), (15, 15, 15)),
    "commercial_transport": ((225, 232, 240), (20, 40, 110)),
    "government": ((240, 240, 240), (150, 20, 20)),
    "other": ((230, 230, 230), (40, 40, 40)),
}
KINDS = list(STYLES)

In [ ]:
def fit_text(draw, box, text, font_path):
    """Largest font size that fits the box, text centred."""
    x0, y0, x1, y1 = box
    size = int((y1 - y0) * 0.95)
    while size > 8:
        font = ImageFont.truetype(str(font_path), size)
        l, t, r, b = draw.textbbox((0, 0), text, font=font)
        if r - l <= (x1 - x0) * 0.92 and b - t <= (y1 - y0) * 0.92:
            break
        size -= 2
    l, t, r, b = draw.textbbox((0, 0), text, font=font)
    return font, (x0 + (x1 - x0 - (r - l)) / 2 - l, y0 + (y1 - y0 - (b - t)) / 2 - t)

def render_plate(code: str, number: str, kind: str, arabic_font: Path, latin_font: Path) -> Image.Image:
    bg, ink = STYLES[kind]
    w, h = CANONICAL_W, CANONICAL_H
    plate = Image.new("RGB", (w, h), bg)
    d = ImageDraw.Draw(plate)
    strip_w = int(0.18 * w)
    d.rectangle([0, 0, strip_w, h], fill=(238, 243, 248))
    f, xy = fit_text(d, (4, 10, strip_w - 4, 50), "الكويت", arabic_font); d.text(xy, "الكويت", font=f, fill=ink)
    f, xy = fit_text(d, (4, 52, strip_w - 4, 84), "KUWAIT", latin_font); d.text(xy, "KUWAIT", font=f, fill=ink)
    for k, colour in enumerate([(0, 122, 61), (255, 255, 255), (206, 17, 38)]):             # flag accent
        d.rectangle([10 + k * (strip_w - 20) // 3, 92, 10 + (k + 1) * (strip_w - 20) // 3, 98], fill=colour)
    for name, text, font in [("code_ar", to_arabic_digits(code), arabic_font), ("code_lat", code, latin_font),
                             ("num_ar", to_arabic_digits(number), arabic_font), ("num_lat", number, latin_font)]:
        f, xy = fit_text(d, cell_box_px(name), text, font)
        d.text(xy, text, font=f, fill=ink)
    d.line([(int(0.37 * w), 8), (int(0.37 * w), h - 8)], fill=ink, width=3)                  # code | number divider
    d.rectangle([1, 1, w - 2, h - 2], outline=ink, width=4)
    return plate

def random_text():
    code = str(random.choices(range(1, 100), weights=[3 if c < 20 else 1 for c in range(1, 100)])[0])
    number = str(random.randint(1, 99999)) if random.random() < 0.9 else str(random.randint(1, 999))
    return code, number

c, n = random_text()
render_plate(c, n, "private", random.choice(ARABIC_FONTS), random.choice(LATIN_FONTS))

In [ ]:
import albumentations as A
import cv2

cell_aug = A.Compose([
    A.Affine(rotate=(-3, 3), shear=(-4, 4), translate_percent=(-0.04, 0.04), p=0.8),
    A.OneOf([A.MotionBlur(blur_limit=5), A.GaussianBlur(blur_limit=(3, 5)), A.Defocus(radius=(1, 2))], p=0.5),
    A.OneOf([A.RandomGamma(gamma_limit=(40, 80)), A.RandomBrightnessContrast(-0.4, 0.3)], p=0.6),   # night / dusk
    A.GaussNoise(p=0.4), A.ImageCompression(quality_range=(30, 90), p=0.5),
])
scene_aug = A.Compose([
    A.RandomRain(p=0.15), A.RandomSunFlare(src_radius=80, p=0.08), A.RandomGamma(gamma_limit=(40, 120), p=0.5),
    A.MotionBlur(blur_limit=7, p=0.3), A.ImageCompression(quality_range=(40, 95), p=0.5),
], keypoint_params=A.KeypointParams(format="xy", remove_invisible=False))

BACKGROUNDS = sorted((DATA / "Plates" / "Backgrounds").glob("*.jpg"))   # car / bay photos without plates, optional

def place_on_scene(plate: np.ndarray, size=640):
    """Paste the plate with a random perspective on a background; return image + corners (xy, pixels)."""
    scene = cv2.imread(str(random.choice(BACKGROUNDS))) if BACKGROUNDS else np.full((size, size, 3), random.randint(40, 200), np.uint8)
    scene = cv2.resize(scene, (size, size))
    pw = random.randint(140, 420); ph = int(pw * CANONICAL_H / CANONICAL_W)
    x, y = random.randint(0, size - pw), random.randint(0, size - ph)
    jitter = lambda: random.uniform(-0.12, 0.12) * ph
    dst = np.float32([[x + jitter(), y + jitter()], [x + pw + jitter(), y + jitter()], [x + pw + jitter(), y + ph + jitter()], [x + jitter(), y + ph + jitter()]])
    src = np.float32([[0, 0], [CANONICAL_W, 0], [CANONICAL_W, CANONICAL_H], [0, CANONICAL_H]])
    m = cv2.getPerspectiveTransform(src, dst)
    warped = cv2.warpPerspective(plate, m, (size, size))
    mask = cv2.warpPerspective(np.full(plate.shape[:2], 255, np.uint8), m, (size, size))
    scene[mask > 0] = warped[mask > 0]
    out = scene_aug(image=scene, keypoints=dst.tolist())
    return out["image"], out["keypoints"]

def make_dataset(count=100_000, start=0, scene_every=4):
    cell_rows, det_rows = [], []
    for i in range(start, start + count):
        code, number = random_text()
        kind = random.choices(KINDS, weights=[70, 10, 10, 5, 5])[0]
        plate = np.array(render_plate(code, number, kind, random.choice(ARABIC_FONTS), random.choice(LATIN_FONTS)))
        texts = {"code_ar": to_arabic_digits(code), "code_lat": code, "num_ar": to_arabic_digits(number), "num_lat": number}
        for name, text in texts.items():
            x0, y0, x1, y1 = cell_box_px(name)
            cell = cell_aug(image=plate[y0:y1, x0:x1])["image"]
            cell = cv2.resize(cv2.cvtColor(cell, cv2.COLOR_RGB2GRAY), (CELL_W, CELL_H))
            file = f"syn_{i:07d}_{name}.png"
            cv2.imwrite(str(OUT / "cells" / file), cell)
            cell_rows.append({"file": f"cells/{file}", "cell": name, "text": text, "kind": kind})
        if i % scene_every == 0:                                    # fewer detector scenes than cell crops
            scene, corners = place_on_scene(cv2.cvtColor(plate, cv2.COLOR_RGB2BGR))
            s = scene.shape[0]
            xs, ys = [c[0] for c in corners], [c[1] for c in corners]
            box = ((min(xs) + max(xs)) / 2 / s, (min(ys) + max(ys)) / 2 / s, (max(xs) - min(xs)) / s, (max(ys) - min(ys)) / s)
            kp = " ".join(f"{cx / s:.6f} {cy / s:.6f} 2" for cx, cy in corners)
            name = f"syn_{i:07d}"
            cv2.imwrite(str(OUT / "detection" / "images" / f"{name}.jpg"), scene)
            (OUT / "detection" / "labels" / f"{name}.txt").write_text(f"{KINDS.index(kind)} " + " ".join(f"{b:.6f}" for b in box) + f" {kp}")
            det_rows.append({"image": f"{name}.jpg", "key": f"{code}-{number}", "kind": kind})
    return pd.DataFrame(cell_rows), pd.DataFrame(det_rows)

cells, scenes = make_dataset(count=2_000)   # raise to 100_000+ on GPU machines
cells.to_csv(OUT / "labels.csv", index=False)
scenes.to_csv(OUT / "detection" / "index.csv", index=False)
cells.head()